# Test YOLO on Tooth Detection Dataset

This notebook loads the fine-tuned PyTorch Lightning model from our training script and evaluates it on the test dataset.

In [9]:
from __future__ import annotations

import os
import sys
import subprocess
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

import lightning as L
from torchmetrics.detection.mean_ap import MeanAveragePrecision

# Ensure project imports work from notebook location.
PROJECT_ROOT = Path('/work')
SCRIPTS_ROOT = PROJECT_ROOT / 'scripts'
for p in (PROJECT_ROOT, SCRIPTS_ROOT):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))

import config
from detection_pipeline import (
    DetectionDownloadConfig,
    ToothDetectionDataset,
    AugmentedToothDetectionDataset,
    build_detection_records,
    build_detection_train_pipeline,
    load_or_download_detection_dataset,
    split_grouped_records
)

In [10]:
@dataclass
class TrainConfig:
    image_size: int = 640
    batch_size: int = 32
    num_workers: int = 0
    conf_threshold: float = 0.001
    iou_threshold: float = 0.6
    pretrained_weights: str = 'yolov5s.pt'
    experiment_name: str = 'label-loss-augmented'
    force_download: bool = False
    
    # Other parameters are ignored during testing but are kept for module loading compatibility
    max_epochs: int = 20
    lr: float = 0.00045636389024364654
    weight_decay: float = 0.000005312109908408371
    adamw_beta1: float = 0.9
    adamw_beta2: float = 0.999
    train_subset_size: int | None = None
    val_subset_size: int | None = None
    test_subset_size: int | None = None

cfg = TrainConfig()
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

Device: cuda


In [11]:
# Clone YOLOv5 if missing and put it first in import order for yolo-specific utils.
YOLOV5_DIR = PROJECT_ROOT / 'external' / 'yolov5'
YOLOV5_DIR.parent.mkdir(parents=True, exist_ok=True)

if not YOLOV5_DIR.exists():
    subprocess.run([
        'git', 'clone', '--depth', '1', 'https://github.com/ultralytics/yolov5.git', str(YOLOV5_DIR)
    ], check=True)

if str(YOLOV5_DIR) not in sys.path:
    sys.path.insert(0, str(YOLOV5_DIR))

from models.yolo import Model
from utils.loss import ComputeLoss
from utils.general import intersect_dicts, non_max_suppression

print('YOLOv5 code loaded from:', YOLOV5_DIR)

YOLOv5 code loaded from: /work/external/yolov5


In [12]:
class YoloTargetAdapterDataset(Dataset):
    """Wraps ToothDetectionDataset to emit labels compatible with YOLOv5 training loss."""

    def __init__(self, base_dataset: Dataset):
        self.base_dataset = base_dataset

    def __len__(self) -> int:
        return len(self.base_dataset)

    def __getitem__(self, index: int):
        image, target = self.base_dataset[index]
        return image, target


def yolo_collate_fn(batch: list[tuple[torch.Tensor, dict[str, Any]]]):
    images = []
    yolo_targets = []
    metric_targets = []

    for i, (img, tgt) in enumerate(batch):
        images.append(img.float())

        boxes_xyxy = tgt['boxes'].float()
        labels_one_based = tgt['labels'].long()

        # TorchMetrics expects class IDs starting at 0.
        metric_targets.append({
            'boxes': boxes_xyxy,
            'labels': (labels_one_based - 1).clamp_min(0),
        })

        if boxes_xyxy.numel() == 0:
            continue

        _, h, w = img.shape
        x1, y1, x2, y2 = boxes_xyxy[:, 0], boxes_xyxy[:, 1], boxes_xyxy[:, 2], boxes_xyxy[:, 3]

        cx = ((x1 + x2) * 0.5) / w
        cy = ((y1 + y2) * 0.5) / h
        bw = (x2 - x1) / w
        bh = (y2 - y1) / h

        cls = (labels_one_based - 1).float().clamp_min(0)
        batch_idx = torch.full((boxes_xyxy.shape[0],), float(i), dtype=torch.float32)

        packed = torch.stack([batch_idx, cls, cx, cy, bw, bh], dim=1)
        yolo_targets.append(packed)

    images = torch.stack(images, dim=0)
    if yolo_targets:
        yolo_targets = torch.cat(yolo_targets, dim=0)
    else:
        yolo_targets = torch.zeros((0, 6), dtype=torch.float32)

    return images, yolo_targets, metric_targets


class ToothDetectionDataModule(L.LightningDataModule):
    def __init__(self, cfg: TrainConfig):
        super().__init__()
        self.cfg = cfg
        self.train_ds = None
        self.val_ds = None
        self.test_ds = None

    def setup(self, stage: str | None = None):
        try:
            from dotenv import load_dotenv
            env_path = PROJECT_ROOT / '.env'
            if load_dotenv is not None and env_path.exists():
                load_dotenv(env_path, override=True)
        except ImportError:
            pass
            
        _, coco_data, image_dirs = load_or_download_detection_dataset(
            DetectionDownloadConfig(api_key=os.getenv('ROBOFLOW_API_KEY')),
            force_download=self.cfg.force_download,
        )
        records, _ = build_detection_records(coco_data, image_dirs)

        train_rec, val_rec, test_rec = split_grouped_records(
            records,
            train_size=config.TRAIN_RATIO,
            val_size=config.VAL_RATIO,
            test_size=config.TEST_RATIO,
            random_state=42
        )

        if self.cfg.train_subset_size is not None:
            train_rec = train_rec[: self.cfg.train_subset_size]
        if self.cfg.val_subset_size is not None:
            val_rec = val_rec[: self.cfg.val_subset_size]
        if self.cfg.test_subset_size is not None:
            test_rec = test_rec[: self.cfg.test_subset_size]

        base_train = ToothDetectionDataset(
            train_rec, image_size=self.cfg.image_size, output_channels=3
        )
        aug_train = AugmentedToothDetectionDataset(base_train, build_detection_train_pipeline())
        
        base_val = ToothDetectionDataset(
            val_rec, image_size=self.cfg.image_size, output_channels=3
        )
        base_test = ToothDetectionDataset(
            test_rec, image_size=self.cfg.image_size, output_channels=3
        )

        self.train_ds = YoloTargetAdapterDataset(aug_train)
        self.val_ds = YoloTargetAdapterDataset(base_val)
        self.test_ds = YoloTargetAdapterDataset(base_test)

        print(f'Train/Val/Test sizes: {len(self.train_ds)}, {len(self.val_ds)}, {len(self.test_ds)}')

    def _loader_kwargs(self) -> dict[str, Any]:
        num_workers = int(self.cfg.num_workers)
        kwargs: dict[str, Any] = {
            'num_workers': num_workers,
            'pin_memory': torch.cuda.is_available(),
            'collate_fn': yolo_collate_fn,
        }
        if num_workers > 0:
            kwargs['persistent_workers'] = True
            kwargs['prefetch_factor'] = 2
        return kwargs

    def train_dataloader(self):
        return DataLoader(
            self.train_ds,
            batch_size=self.cfg.batch_size,
            shuffle=True,
            drop_last=True,
            **self._loader_kwargs(),
        )

    def val_dataloader(self):
        return DataLoader(
            self.val_ds,
            batch_size=self.cfg.batch_size,
            shuffle=False,
            **self._loader_kwargs(),
        )

    def test_dataloader(self):
        return DataLoader(
            self.test_ds,
            batch_size=self.cfg.batch_size,
            shuffle=False,
            **self._loader_kwargs(),
        )

In [13]:
class LitYOLOv5(L.LightningModule):
    def __init__(self, cfg: TrainConfig, num_classes: int = 32):
        super().__init__()
        self.save_hyperparameters(ignore=["cfg"])

        self.cfg = cfg
        self.num_classes = num_classes
        self.compute_loss = None

        # When testing, we can initialize with either base weights or just state structure
        ckpt_path = Path(cfg.pretrained_weights)
        if ckpt_path.exists():
            ckpt = torch.load(ckpt_path, map_location="cpu")
            yolo_cfg = ckpt["model"].yaml
        else:
            # Fallback configuration
            yolo_cfg = YOLOV5_DIR / "models" / "yolov5s.yaml"

        self.model = Model(yolo_cfg, ch=3, nc=num_classes).float()
        self.model.hyp = {
            "box": 0.05, "cls": 0.3, "obj": 0.7,
            "cls_pw": 1.0, "obj_pw": 1.0, "fl_gamma": 0.0,
            "label_smoothing": 0.0, "anchor_t": 4.0,
        }

        self.map_metric = MeanAveragePrecision(
            box_format="xyxy",
            class_metrics=False,
        )

    def on_train_start(self):
        pass # Ignored for testing

    def forward(self, x: torch.Tensor):
        return self.model(x)

    def training_step(self, batch, batch_idx: int):
        pass # Ignored for testing 

    def validation_step(self, batch, batch_idx: int):
        pass # Not using validation here

    def on_validation_epoch_end(self):
        pass # Not using validation here

    def test_step(self, batch, batch_idx: int):
        images, _, metric_targets = batch
        raw_output = self.model(images)
        preds = raw_output[0] if isinstance(raw_output, (tuple, list)) else raw_output

        nms_preds = non_max_suppression(
            preds,
            conf_thres=self.cfg.conf_threshold,
            iou_thres=self.cfg.iou_threshold,
            multi_label=False,
            max_det=300,
        )

        metric_preds = []
        for det in nms_preds:
            if det is None or len(det) == 0:
                metric_preds.append({
                    "boxes": torch.zeros((0, 4), device=self.device),
                    "scores": torch.zeros((0,), device=self.device),
                    "labels": torch.zeros((0,), dtype=torch.long, device=self.device),
                })
                continue
            metric_preds.append({
                "boxes": det[:, :4],
                "scores": det[:, 4],
                "labels": det[:, 5].long(),
            })

        metric_targets_list = [
            {
                "boxes": target["boxes"].to(self.device),
                "labels": target["labels"].to(self.device),
            }
            for target in metric_targets
        ]

        self.map_metric.update(metric_preds, metric_targets_list)

    def on_test_epoch_end(self):
        metrics = self.map_metric.compute()

        self.log("test/map", metrics["map"], prog_bar=True)
        self.log("test/map_50", metrics["map_50"], prog_bar=True)
        self.log("test/map_75", metrics["map_75"])

        self.map_metric.reset()

    def configure_optimizers(self):
        pass

In [14]:
# Setup data module
datamodule = ToothDetectionDataModule(cfg)
datamodule.setup('test')

print(f'\nDataset sizes:')
print(f'  Test:  {len(datamodule.test_ds)} samples')

Train/Val/Test sizes: 528, 66, 67

Dataset sizes:
  Test:  67 samples


### Visualize Ground Truth Bounding Boxes

Let's verify our data by drawing the ground truth bounding boxes over samples from each of our dataset splits (Train, Validation, and Test).

In [15]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

def visualize_dataset_samples(dataset, title: str, num_samples: int = 3):
    """
    Plots a few sample images from the given YOLO adapted dataset 
    with their ground truth bounding boxes overlaid.
    """
    fig, axes = plt.subplots(1, num_samples, figsize=(5 * num_samples, 5))
    fig.suptitle(title, fontsize=16)
    
    # In case of num_samples=1, axes is not an array
    if num_samples == 1:
        axes = [axes]
        
    for i in range(num_samples):
        # The dataset returns (image_tensor, target_dict)
        img_tensor, target = dataset[i]
        
        # Convert CxHxW tensor in [0, 1] to HxWxC numpy array
        img_np = img_tensor.detach().cpu().numpy().transpose((1, 2, 0))
        img_np = np.clip(img_np, 0, 1)
        
        ax = axes[i]
        ax.imshow(img_np)
        ax.axis('off')
        
        if target is not None and 'boxes' in target and target['boxes'].numel() > 0:
            boxes = target['boxes'].detach().cpu().numpy()
            labels = target['labels'].detach().cpu().numpy()
            
            for box, label in zip(boxes, labels):
                x1, y1, x2, y2 = box
                width, height = x2 - x1, y2 - y1
                
                # Draw the bounding box
                rect = patches.Rectangle(
                    (x1, y1), width, height, 
                    linewidth=2, edgecolor='red', facecolor='none'
                )
                ax.add_patch(rect)
                
                # Add a label background for visibility
                ax.text(
                    x1, max(0, y1 - 2), f"Class {label}", 
                    color='white', fontsize=10, weight='bold',
                    bbox=dict(facecolor='red', alpha=0.5, edgecolor='none', pad=1)
                )
                
    plt.tight_layout()
    plt.show()

# Setup is needed for train and val splits since datamodule.setup('test') only checks what's required,
# but our `ToothDetectionDataModule.setup()` actually loads all three splits.
# We visualize 3 samples from each split:
visualize_dataset_samples(datamodule.train_ds, "Training Data Annotations")
visualize_dataset_samples(datamodule.val_ds, "Validation Data Annotations")
visualize_dataset_samples(datamodule.test_ds, "Test Data Annotations")

In [16]:
# Load the final model
model_path = PROJECT_ROOT / 'output' / cfg.experiment_name / 'weights' / 'final_model.pt'

if not model_path.exists():
    raise FileNotFoundError(f"Model weights not found at {model_path}. Did you run the training script successfully?")

print(f"Loading weights from {model_path}...")

model_instance = LitYOLOv5(cfg=cfg, num_classes=32)

# Using strict=False as we saved raw YOLOv5 states. 
# They need to be loaded into the PyTorch Lightning wrapped YOLO model.
model_instance.model.load_state_dict(torch.load(model_path, map_location='cpu'), strict=False)

print("Loaded model correctly!")

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
Overriding model.yaml nc=80 with nc=32

                 from  n    params  module                                  arguments                   

Loading weights from /work/output/label-loss-augmented/weights/final_model.pt...


Model summary: 214 layers, 7105933 parameters, 7105933 gradients, 16.2 GFLOPs

You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.


Loaded model correctly!


In [17]:
# Initialize lightning trainer
trainer = L.Trainer(
    accelerator='auto',
    devices=1,
    logger=False  # Typically disabled for pure testing.
)

# Run standard PyTorch Lightning testing script
print("Running prediction on the test dataset...")
test_results = trainer.test(model_instance, datamodule=datamodule)

print("\n--- Test Results ---")
for key, val in test_results[0].items():
    print(f"{key}: {val:.4f}")

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
You are using a CUDA device ('NVIDIA GeForce RTX 4070') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision


Running prediction on the test dataset...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


Train/Val/Test sizes: 528, 66, 67


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test/map          │    0.4065264165401459     │
│        test/map_50        │    0.5789527893066406     │
│        test/map_75        │    0.48352891206741333    │
└───────────────────────────┴───────────────────────────┘


--- Test Results ---
test/map: 0.4065
test/map_50: 0.5790
test/map_75: 0.4835
